# 04 — Weapon Comparison

Compare two weapons on the same character against the same NPC target.

Edit the **Configuration** cell to change:
- Attacker class, stats, skill level
- The two weapons to compare
- The NPC target stats and armor

In [ ]:
from pathlib import Path

from omega.model.constants import (
    SKILLID_ANATOMY, SKILLID_ENTICEMENT, SKILLID_FENCING,
    SKILLID_MUSICIANSHIP, SKILLID_PEACEMAKING, SKILLID_PROVOCATION,
    SKILLID_SWORDSMANSHIP, SKILLID_TACTICS,
)
from omega.shard import ShardData
from omega.simulation import (
    ArmorSpec, CombatantSpec, Scenario, WeaponSpec, run_scenario,
)
from omega.reporting.tables import comparison_table, format_table_html
from omega.reporting.plots import comparison_breakdown, comparison_overlay

SHARD_ROOT = Path("submodules/zuluhotel_omega_2.5")
if not SHARD_ROOT.exists():
    SHARD_ROOT = Path("../submodules/zuluhotel_omega_2.5")
shard = ShardData.from_path(SHARD_ROOT)
parse_results = shard.parse_combat_scripts()

## Configuration

In [ ]:
# --- Attacker: Bladesinger with all in-class skills ---
SKILL_LEVEL = 100
BLADESINGER_SKILLS = {
    SKILLID_ANATOMY: SKILL_LEVEL,
    SKILLID_ENTICEMENT: SKILL_LEVEL,
    SKILLID_FENCING: SKILL_LEVEL,
    SKILLID_MUSICIANSHIP: SKILL_LEVEL,
    SKILLID_PEACEMAKING: SKILL_LEVEL,
    SKILLID_PROVOCATION: SKILL_LEVEL,
    SKILLID_SWORDSMANSHIP: SKILL_LEVEL,
    SKILLID_TACTICS: SKILL_LEVEL,
}

ATTACKER_STATS = {"str": 100, "dex": 100, "int": 25}
CLASS_LEVEL = 5

# --- Two weapons to compare ---
WEAPON_A = WeaponSpec(name="Broadsword", damage="3d6+2")
WEAPON_B = WeaponSpec(name="Katana", damage="2d8+4")

# --- NPC target ---
DEFENDER = CombatantSpec(
    name="Target", is_npc=True,
    str_=50, dex_=50, int_=50, hp=500,
    armor=ArmorSpec(ar=30),
)

# --- Simulation settings ---
ITERATIONS = 200
BASE_SEED = 42

print(f"Bladesinger (level {CLASS_LEVEL}, skills @ {SKILL_LEVEL})")
print(f"  STR={ATTACKER_STATS['str']}  DEX={ATTACKER_STATS['dex']}  INT={ATTACKER_STATS['int']}")
print(f"Weapon A: {WEAPON_A.name} ({WEAPON_A.damage})")
print(f"Weapon B: {WEAPON_B.name} ({WEAPON_B.damage})")
print(f"Target: AR {DEFENDER.armor.ar}, HP {DEFENDER.hp}")

## Run Simulation

In [ ]:
RUN_KW = dict(
    parse_results=parse_results,
    config_resolver=shard.resolve_config_path,
    em_modules_dir=shard.root / "scripts" / "modules",
)

def make_attacker(weapon):
    return CombatantSpec(
        name=f"Bladesinger ({weapon.name})",
        skills=BLADESINGER_SKILLS,
        str_=ATTACKER_STATS["str"],
        dex_=ATTACKER_STATS["dex"],
        int_=ATTACKER_STATS["int"],
        class_levels={"IsBladesinger": CLASS_LEVEL},
        weapon=weapon,
    )

result_a = run_scenario(
    Scenario(attacker=make_attacker(WEAPON_A), defender=DEFENDER,
             iterations=ITERATIONS, base_seed=BASE_SEED),
    **RUN_KW,
)
result_b = run_scenario(
    Scenario(attacker=make_attacker(WEAPON_B), defender=DEFENDER,
             iterations=ITERATIONS, base_seed=BASE_SEED),
    **RUN_KW,
)

results = {WEAPON_A.name: result_a, WEAPON_B.name: result_b}

for label, r in results.items():
    ds = r.damage_stats
    print(f"  {label:20s}  mean={ds.mean:6.2f}  median={ds.median:6.2f}")

In [ ]:
# Overlaid damage distributions
comparison_overlay(
    results,
    title=f"Weapon Comparison — {WEAPON_A.name} vs {WEAPON_B.name}",
)

In [ ]:
# Stacked bar: base damage, absorbed, final
comparison_breakdown(
    results,
    title=f"Damage Breakdown — {WEAPON_A.name} vs {WEAPON_B.name}",
)

In [ ]:
# Side-by-side comparison table
from IPython.display import HTML

rows = comparison_table(
    results,
    stats=["mean", "median", "min", "max", "p5", "p95", "total", "hit_rate", "absorbed_mean"],
)
HTML(format_table_html(rows))